# 02 — Ingesta, vocabulario de dominio, embeddings y grafo

**Checkpoint: Día 1 — revisión inicial.** Construye todo `processed/` desde los datos crudos. Cada celda corresponde a una etapa del pipeline (`arquitectura_saberlink-md.md`). Nada aquí modifica los archivos originales — verificado al final con un hash antes/después.

In [1]:
import sys
sys.path.insert(0, '..')

import hashlib
from saberlink import config, ingest, domain_vocab, graph_build, vector_store

## Hash de los datos crudos, antes de tocar nada

In [2]:
def hash_raw_data():
    return {str(p): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(config.DATA_ROOT.rglob('*')) if p.is_file()}

hashes_before = hash_raw_data()
print(f'{len(hashes_before)} archivos crudos con hash calculado')

89 archivos crudos con hash calculado


## Etapa A1 — Ingesta: `entities.parquet` + `fields_index.parquet`

In [3]:
entities, fields_index = ingest.run()
print('entities:', entities.shape)
print('fields_index:', fields_index.shape)
entities['entity_type'].value_counts()

entities: (3267, 96)
fields_index: (10756, 7)


entity_type
EXP     720
THS     650
LO      378
PUB     360
PRJ     320
COM     252
INV     180
SUB     126
CAP      96
LIN      60
NEED     42
SRC      35
GRP      24
PRG      18
FAC       6
Name: count, dtype: int64

## Etapa A2 — Vocabulario de dominio (con frecuencia de documento para ponderar por especificidad)

In [4]:
vocab = domain_vocab.build_vocab(entities)
doc_freq = domain_vocab.build_document_frequency(entities, vocab)
domain_vocab.save_vocab(vocab, doc_freq)
print(f'{len(vocab)} términos de dominio')
print('ejemplos más específicos (baja frecuencia):',
      sorted(doc_freq.items(), key=lambda x: x[1])[:8])
print('ejemplos más genéricos (alta frecuencia):',
      sorted(doc_freq.items(), key=lambda x: -x[1])[:8])

361 términos de dominio
ejemplos más específicos (baja frecuencia): [('automatización', 1), ('Transformación digital', 1), ('Innovación educativa', 1), ('Salud pública y datos', 1), ('Learning analytics', 1), ('Agua y sostenibilidad', 1), ('Matemáticas aplicadas', 1), ('Física e instrumentación', 1)]
ejemplos más genéricos (alta frecuencia): [('herramientas', 474), ('series temporales', 186), ('optimización', 140), ('infraestructura', 135), ('procesamiento de señales', 117), ('simulación', 113), ('Ambiental', 105), ('Optimización', 97)]


## Etapa A3 — Embeddings por campo → ChromaDB
Modelo `paraphrase-multilingual-MiniLM-L12-v2`. Esto puede tardar uno o dos minutos la primera vez (descarga del modelo + embebido de ~10.7k campos).

In [5]:
n = vector_store.run()
print(f'{n} vectores indexados en la colección "{config.CHROMA_COLLECTION_NAME}"')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

10725 vectores indexados en la colección "saberlink_fields"


## Etapa A4 — Grafo de relaciones explícitas (networkx)

In [6]:
graph = graph_build.run()
print(f'{graph.number_of_nodes()} nodos, {graph.number_of_edges()} aristas')
# Ejemplo: de dónde viene una arista
print(graph['PRG-001']['FAC-001'])

3267 nodos, 6431 aristas
{'relation_type': 'PRG.faculty_id', 'edge_kind': 'explicit'}


## Verificación final: los datos crudos no cambiaron

In [7]:
hashes_after = hash_raw_data()
assert hashes_before == hashes_after, 'los datos originales fueron modificados!'
print('OK: ningún archivo original fue modificado por la ingesta.')

OK: ningún archivo original fue modificado por la ingesta.
